# 02 - Normalización y limpieza inicial

**Objetivo.** Normalizar textos, categorías, barrios, monedas, operaciones, fechas y valores inválidos; construir versiones preprocesadas por fuente/operación y una base consolidada con columnas canónicas.

La limpieza se realiza por dataset para no mezclar problemas específicos de una fuente con faltantes estructurales de otra.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

## Carga de archivos raw

In [ ]:
INVALID_MARKERS = {
    '', ' ', 'nan', 'NaN', 'NAN', 'none', 'None', 'NONE', 'null', 'NULL',
    's/d', 'S/D', 'sd', 'SD', 'sin dato', 'Sin dato', 'Sin Datos', 'sin datos',
    'no informa', 'No informa', 'no informado', 'No informado', 'n/a', 'N/A',
    '-', '--', '---', '?', '99999', '-99999', '-9999', '9999'
}

RAW_SOURCES = {
    'df_raw_meli_alq': {
        'descripcion': 'Mercado Libre - alquiler permanente',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_alquiler').glob('meli_alq_*.csv')),
    },
    'df_raw_meli_alq_temp': {
        'descripcion': 'Mercado Libre - alquiler temporario',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_alquiler temporario').glob('meli_alq_temp_*.csv')),
    },
    'df_raw_meli_vtas': {
        'descripcion': 'Mercado Libre - venta',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_ventas').glob('meli_vtas_*.csv')),
    },
    'df_raw_argenprop': {
        'descripcion': 'Argenprop - publicaciones extraidas',
        'paths': [RAW_DIR / 'argenprop' / 'propiedades_argenprop.csv'],
    },
    'df_raw_zonaprop': {
        'descripcion': 'Zonaprop - publicaciones extraidas',
        'paths': [RAW_DIR / 'zonaprop' / 'propiedades_zonaprop.csv'],
    },
    'df_raw_airbnb_norm': {
        'descripcion': 'Airbnb CABA normalizado mensual. Base prioritaria para analisis comparable',
        'paths': [RAW_DIR / 'airbnb' / 'airbnb_caba_normalizado_mensual.csv'],
    },
}


def read_csv_safely(path):
    return pd.read_csv(path, low_memory=False)


def load_many(paths, source_name):
    existing = [Path(p) for p in paths if Path(p).exists()]
    if not existing:
        print(f'ATENCION: no se encontraron archivos para {source_name}')
        return pd.DataFrame()
    frames = []
    for path in existing:
        df = read_csv_safely(path)
        df['_archivo_origen'] = str(path.relative_to(REPO_ROOT))
        frames.append(df)
    return pd.concat(frames, ignore_index=True, sort=False)

raw_dfs = {}
for df_name, spec in RAW_SOURCES.items():
    raw_dfs[df_name] = load_many(spec['paths'], df_name)
    globals()[df_name] = raw_dfs[df_name]
    print(f"{df_name}: {raw_dfs[df_name].shape} | {spec['descripcion']}")

## Funciones de limpieza, normalización y consolidación

In [ ]:
CABA_BARRIOS = {
    'agronomia':'agronomia', 'almagro':'almagro', 'balvanera':'balvanera', 'barracas':'barracas',
    'belgrano':'belgrano', 'boedo':'boedo', 'caballito':'caballito', 'chacarita':'chacarita',
    'coghlan':'coghlan', 'colegiales':'colegiales', 'constitucion':'constitucion', 'flores':'flores',
    'floresta':'floresta', 'la boca':'la boca', 'liniers':'liniers', 'mataderos':'mataderos',
    'monserrat':'monserrat', 'monte castro':'monte castro', 'nueva pompeya':'nueva pompeya',
    'nunez':'nunez', 'palermo':'palermo', 'parque avellaneda':'parque avellaneda',
    'parque chacabuco':'parque chacabuco', 'parque chas':'parque chas',
    'parque patricios':'parque patricios', 'paternal':'paternal', 'puerto madero':'puerto madero',
    'recoleta':'recoleta', 'retiro':'retiro', 'saavedra':'saavedra', 'san cristobal':'san cristobal',
    'san nicolas':'san nicolas', 'san telmo':'san telmo', 'velez sarsfield':'velez sarsfield',
    'versalles':'versalles', 'villa crespo':'villa crespo', 'villa del parque':'villa del parque',
    'villa devoto':'villa devoto', 'villa general mitre':'villa general mitre', 'villa lugano':'villa lugano',
    'villa luro':'villa luro', 'villa ortuzar':'villa ortuzar', 'villa pueyrredon':'villa pueyrredon',
    'villa real':'villa real', 'villa riachuelo':'villa riachuelo', 'villa santa rita':'villa santa rita',
    'villa soldati':'villa soldati', 'villa urquiza':'villa urquiza'
}

BARRIO_ALIASES = {
    'nuñez': 'nunez', 'nunez': 'nunez', 'agronomía': 'agronomia',
    'san nicolás': 'san nicolas', 'san cristóbal': 'san cristobal',
    'vélez sársfield': 'velez sarsfield', 'villa gral. mitre': 'villa general mitre',
    'villa gral mitre': 'villa general mitre', 'palermo chico': 'palermo',
    'palermo hollywood': 'palermo', 'palermo soho': 'palermo', 'las cañitas': 'palermo',
    'belgrano chico': 'belgrano', 'barrio norte': 'recoleta',
    'capital federal': np.nan, 'ciudad autonoma de buenos aires': np.nan,
    'ciudad autónoma de buenos aires': np.nan, 'caba': np.nan,
}


def strip_accents(value):
    if pd.isna(value):
        return np.nan
    text = unicodedata.normalize('NFKD', str(value))
    return ''.join(ch for ch in text if not unicodedata.combining(ch))


def clean_text(value, lower=True):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('\xa0', ' ').strip()
    text = re.sub(r'\s+', ' ', text)
    if text in INVALID_MARKERS:
        return np.nan
    if lower:
        text = strip_accents(text).lower()
    return text


def normalize_barrio(value):
    text = clean_text(value, lower=True)
    if pd.isna(text):
        return np.nan
    text = BARRIO_ALIASES.get(text, text)
    if pd.isna(text):
        return np.nan
    return CABA_BARRIOS.get(text, text)


def normalize_operation(value):
    text = clean_text(value, lower=True)
    if pd.isna(text):
        return np.nan
    if 'tempor' in text:
        return 'alquiler_temporal'
    if 'alquiler' in text:
        return 'alquiler'
    if 'venta' in text:
        return 'venta'
    return text.replace(' ', '_')


def normalize_currency(value):
    text = clean_text(value, lower=False)
    if pd.isna(text):
        return np.nan
    t = strip_accents(str(text)).upper().replace('$', 'ARS').strip()
    if t in {'US$', 'U$S', 'USD'}:
        return 'USD'
    if t in {'ARS', 'PESOS', 'PESO'}:
        return 'ARS'
    return t


def replace_invalid_markers(df):
    out = df.copy()
    obj_cols = out.select_dtypes(include=['object', 'string']).columns
    for col in obj_cols:
        stripped = out[col].astype('string').str.strip()
        out.loc[stripped.isin(INVALID_MARKERS), col] = np.nan
    return out


def parse_date_columns(df):
    out = df.copy()
    date_like = [c for c in out.columns if any(token in c.lower() for token in ['fecha', 'date', 'scraped_at'])]
    for col in date_like:
        out[col] = pd.to_datetime(out[col], errors='coerce', utc=True)
    return out


def add_null_flags(df, exclude=None):
    out = df.copy()
    exclude = set(exclude or [])
    original_cols = [c for c in out.columns if c not in exclude and not c.endswith('_is_null')]
    for col in original_cols:
        if out[col].isna().any():
            out[f'{col}_is_null'] = out[col].isna().astype('int8')
    return out


def to_numeric(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    return out


def first_existing(df, columns, default=np.nan):
    existing = [c for c in columns if c in df.columns]
    if not existing:
        return pd.Series(default, index=df.index)
    result = df[existing[0]].copy()
    for col in existing[1:]:
        result = result.combine_first(df[col])
    return result


def standardize_meli(df, operacion_forzada=None):
    out = replace_invalid_markers(df)
    out = parse_date_columns(out)
    out = to_numeric(out, ['precio', 'superficie_min_m2', 'superficie_max_m2', 'ambientes_min', 'ambientes_max', 'banios_min', 'banios_max', 'dormitorios_min', 'dormitorios_max', 'comuna', 'precio_m2_min', 'precio_m2_max', 'altura'])
    out['fuente_norm'] = 'mercado_libre'
    out['tipo_operacion_norm'] = operacion_forzada or out.get('tipo_operacion', pd.Series(index=out.index, dtype='object')).map(normalize_operation)
    out['barrio_norm'] = out.get('barrio', pd.Series(index=out.index, dtype='object')).map(normalize_barrio)
    out['moneda_norm'] = out.get('moneda', pd.Series(index=out.index, dtype='object')).map(normalize_currency)
    out['precio_m2_ref'] = first_existing(out, ['precio_m2_min'])
    mask_precio_m2 = out['precio_m2_ref'].isna() & out.get('precio', np.nan).notna() & out.get('superficie_min_m2', np.nan).gt(0)
    out.loc[mask_precio_m2, 'precio_m2_ref'] = out.loc[mask_precio_m2, 'precio'] / out.loc[mask_precio_m2, 'superficie_min_m2']
    out['id_publicacion'] = out.get('item_id')
    out['superficie_ref_m2'] = first_existing(out, ['superficie_min_m2', 'superficie_max_m2'])
    out['ambientes_ref'] = first_existing(out, ['ambientes_min', 'ambientes_max'])
    out['banios_ref'] = first_existing(out, ['banios_min', 'banios_max'])
    out['dormitorios_ref'] = first_existing(out, ['dormitorios_min', 'dormitorios_max'])
    out['fecha_extraccion'] = out.get('scraped_at_utc')
    return out


def standardize_portal(df, fuente_norm):
    out = replace_invalid_markers(df)
    out = parse_date_columns(out)
    out = to_numeric(out, ['precio', 'superficie_total_m2', 'superficie_cubierta_m2', 'ambientes', 'banios', 'dormitorios', 'comuna', 'precio_m2_calculado', 'latitud', 'longitud', 'altura'])
    out['fuente_norm'] = fuente_norm
    out['tipo_operacion_norm'] = out.get('tipo_operacion', pd.Series(index=out.index, dtype='object')).map(normalize_operation)
    out['barrio_norm'] = first_existing(out, ['barrio', 'localidad', 'calle']).map(normalize_barrio)
    out['moneda_norm'] = out.get('moneda', pd.Series(index=out.index, dtype='object')).map(normalize_currency)
    out['precio_m2_ref'] = first_existing(out, ['precio_m2_calculado'])
    mask_precio_m2 = out['precio_m2_ref'].isna() & out.get('precio', np.nan).notna() & out.get('superficie_total_m2', np.nan).gt(0)
    out.loc[mask_precio_m2, 'precio_m2_ref'] = out.loc[mask_precio_m2, 'precio'] / out.loc[mask_precio_m2, 'superficie_total_m2']
    out['id_publicacion'] = out.get('property_id')
    out['superficie_ref_m2'] = first_existing(out, ['superficie_total_m2', 'superficie_cubierta_m2'])
    out['ambientes_ref'] = out.get('ambientes')
    out['banios_ref'] = out.get('banios')
    out['dormitorios_ref'] = out.get('dormitorios')
    out['fecha_extraccion'] = out.get('scraped_at_utc')
    return out


def standardize_airbnb(df):
    out = replace_invalid_markers(df)
    out = parse_date_columns(out)
    out = to_numeric(out, ['precio', 'ambientes', 'banios', 'dormitorios', 'comuna', 'latitud', 'longitud', 'rating', 'reviews', 'capacidad', 'camas'])
    out['fuente_norm'] = 'airbnb'
    out['tipo_operacion_norm'] = 'alquiler_temporal'
    out['barrio_norm'] = first_existing(out, ['barrio_norm', 'barrio']).map(normalize_barrio)
    out['moneda_norm'] = out.get('moneda', pd.Series(index=out.index, dtype='object')).map(normalize_currency)
    out['id_publicacion'] = out.get('property_id').astype('string')
    out['superficie_ref_m2'] = np.nan
    out['ambientes_ref'] = out.get('ambientes')
    out['banios_ref'] = out.get('banios')
    out['dormitorios_ref'] = out.get('dormitorios')
    out['fecha_extraccion'] = out.get('scraped_at_utc')
    out['precio_noche_estimado'] = out['precio']
    out['precio_mensual_estimado'] = np.where(out['precio'].notna(), out['precio'] * 30, np.nan)
    out['precio_m2_ref'] = np.nan
    return out

CANONICAL_COLUMNS = [
    'id_publicacion', 'fuente_norm', 'fuente', 'tipo_operacion_norm', 'tipo_operacion',
    'url', 'fecha_extraccion', 'scraped_at_utc', '_archivo_origen', 'titulo', 'descripcion',
    'tipo_propiedad', 'moneda_norm', 'moneda', 'precio', 'precio_texto', 'precio_m2_ref',
    'precio_noche_estimado', 'precio_mensual_estimado', 'barrio', 'barrio_norm', 'comuna',
    'localidad', 'provincia', 'pais', 'direccion', 'direccion_completa', 'calle', 'altura',
    'latitud', 'longitud', 'superficie_ref_m2', 'superficie_min_m2', 'superficie_max_m2',
    'superficie_total_m2', 'superficie_cubierta_m2', 'ambientes_ref', 'ambientes',
    'ambientes_min', 'ambientes_max', 'dormitorios_ref', 'dormitorios', 'dormitorios_min',
    'dormitorios_max', 'banios_ref', 'banios', 'banios_min', 'banios_max', 'inmobiliaria',
    'seller_id', 'seller_nombre', 'seller_superhost', 'rating', 'reviews', 'apto_credito',
    'balcon', 'balcon_o_patio', 'terraza', 'parrilla', 'pileta', 'cochera',
    'cochera_mencionada', 'amenities_mencionadas', 'precio_sospechoso', 'parse_ok',
    'parse_warnings', 'cantidad_imagenes', 'imagen_principal'
]


def select_canonical(df):
    cols = [c for c in CANONICAL_COLUMNS if c in df.columns]
    return df[cols].copy()

## Revisión de categorías antes de normalizar

In [ ]:
cat_vars_relevantes = ['fuente', 'tipo_operacion', 'tipo_propiedad', 'moneda', 'barrio', 'localidad', 'provincia']
for name, df in raw_dfs.items():
    print('\n' + '=' * 100)
    print(name)
    for col in cat_vars_relevantes:
        if col in df.columns:
            print(f'\n-- {col} --')
            display(df[col].astype('string').str.strip().value_counts(dropna=False).head(25).to_frame('conteo'))

## Revisión y parseo de fechas

In [ ]:
for name, df in raw_dfs.items():
    date_like = [c for c in df.columns if any(token in c.lower() for token in ['fecha', 'date', 'scraped_at'])]
    print(name, date_like)
    for col in date_like:
        parsed = pd.to_datetime(df[col], errors='coerce', utc=True)
        print(f'  {col}: validas={parsed.notna().sum()} | invalidas/nulas={parsed.isna().sum()}')

## Normalización por fuente y operación

Airbnb se trata con cuidado porque ya viene normalizado. No se vuelve a parsear desde cero: se conserva su estructura y se crean variables comparables con el resto de las fuentes.

In [ ]:
df_preprocesado_meli_alq = standardize_meli(df_raw_meli_alq, operacion_forzada='alquiler')
df_preprocesado_meli_alq_temp = standardize_meli(df_raw_meli_alq_temp, operacion_forzada='alquiler_temporal')
df_preprocesado_meli_vtas = standardize_meli(df_raw_meli_vtas, operacion_forzada='venta')
df_preprocesado_argenprop = standardize_portal(df_raw_argenprop, 'argenprop')
df_preprocesado_zonaprop = standardize_portal(df_raw_zonaprop, 'zonaprop')
df_preprocesado_airbnb_temp = standardize_airbnb(df_raw_airbnb_norm)

preprocesados = {
    'df_preprocesado_meli_alq': df_preprocesado_meli_alq,
    'df_preprocesado_meli_alq_temp': df_preprocesado_meli_alq_temp,
    'df_preprocesado_meli_vtas': df_preprocesado_meli_vtas,
    'df_preprocesado_argenprop': df_preprocesado_argenprop,
    'df_preprocesado_zonaprop': df_preprocesado_zonaprop,
    'df_preprocesado_airbnb_temp': df_preprocesado_airbnb_temp,
}
for name, df in preprocesados.items():
    print(name, df.shape)

## Impacto de la transformación de inválidos a nulos

In [ ]:
resumen_nulos_pre = []
for name, df in preprocesados.items():
    resumen_nulos_pre.append({
        'dataframe': name,
        'filas': len(df),
        'columnas': df.shape[1],
        'total_nulos': int(df.isna().sum().sum()),
        'pct_celdas_nulas': round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2) if df.shape[0] and df.shape[1] else np.nan,
    })
resumen_nulos_pre = pd.DataFrame(resumen_nulos_pre)
display(resumen_nulos_pre)

## Consolidación con columnas canónicas

Se conserva una versión preprocesada por fuente y tipo de operación para evitar que los faltantes estructurales entre portales se confundan con problemas de calidad. Luego se construye una base consolidada con variables comunes y flags de fuente/operación, utilizada para el EDA inicial y comparaciones transversales.

In [ ]:
canonical_frames = []
for name, df in preprocesados.items():
    tmp = select_canonical(df)
    tmp['dataset_preprocesado'] = name
    canonical_frames.append(tmp)

df_publicaciones_consolidadas = pd.concat(canonical_frames, ignore_index=True, sort=False)
display(df_publicaciones_consolidadas.head())
print(df_publicaciones_consolidadas.shape)

## Controles rápidos post-consolidación

## Celdas de decisión: duplicados y campos críticos

Estas celdas dejan explícito el criterio de tratamiento. Por defecto **no se eliminan registros**: se generan reportes y se conserva la trazabilidad. Si el grupo decide aplicar una regla, debe cambiar el parámetro correspondiente y justificarlo en el markdown de cierre.

In [ ]:
# Parametros de decision conservadores.
# Cambiar a True solo si el grupo decide aplicar efectivamente el filtro.
ELIMINAR_DUPLICADOS_EXACTOS = False
ELIMINAR_REGISTROS_SIN_CAMPOS_CRITICOS = False

CAMPOS_CRITICOS_CONSOLIDADO = ['id_publicacion', 'fuente_norm', 'tipo_operacion_norm', 'precio', 'moneda_norm', 'barrio_norm']

reporte_decisiones_consolidado = {
    'filas_antes': len(df_publicaciones_consolidadas),
    'duplicados_exactos': int(df_publicaciones_consolidadas.duplicated().sum()),
    'filas_sin_alguno_campos_criticos': int(df_publicaciones_consolidadas[CAMPOS_CRITICOS_CONSOLIDADO].isna().any(axis=1).sum()),
    'eliminar_duplicados_exactos': ELIMINAR_DUPLICADOS_EXACTOS,
    'eliminar_registros_sin_campos_criticos': ELIMINAR_REGISTROS_SIN_CAMPOS_CRITICOS,
}

if ELIMINAR_DUPLICADOS_EXACTOS:
    df_publicaciones_consolidadas = df_publicaciones_consolidadas.drop_duplicates().copy()

if ELIMINAR_REGISTROS_SIN_CAMPOS_CRITICOS:
    df_publicaciones_consolidadas = df_publicaciones_consolidadas.dropna(subset=CAMPOS_CRITICOS_CONSOLIDADO).copy()

reporte_decisiones_consolidado['filas_despues'] = len(df_publicaciones_consolidadas)
reporte_decisiones_consolidado = pd.DataFrame([reporte_decisiones_consolidado])
display(reporte_decisiones_consolidado)
reporte_decisiones_consolidado.to_csv(REPORTS_DIR / 'decisiones_filtros_consolidado_v1.csv', index=False)

In [ ]:
controles = {
    'filas_por_fuente': df_publicaciones_consolidadas['fuente_norm'].value_counts(dropna=False),
    'filas_por_operacion': df_publicaciones_consolidadas['tipo_operacion_norm'].value_counts(dropna=False),
    'monedas': df_publicaciones_consolidadas['moneda_norm'].value_counts(dropna=False),
    'barrios_top_30': df_publicaciones_consolidadas['barrio_norm'].value_counts(dropna=False).head(30),
}
for titulo, serie in controles.items():
    print('\n' + titulo)
    display(serie.to_frame('conteo'))

campos_criticos = ['id_publicacion', 'fuente_norm', 'tipo_operacion_norm', 'precio', 'moneda_norm', 'barrio_norm', 'superficie_ref_m2', 'precio_m2_ref']
display(df_publicaciones_consolidadas[campos_criticos].isna().mean().mul(100).round(2).to_frame('pct_nulos'))

## Guardado de datasets preprocesados

In [ ]:
outputs = {
    'df_preprocesado_meli_alq': df_preprocesado_meli_alq,
    'df_preprocesado_meli_alq_temp': df_preprocesado_meli_alq_temp,
    'df_preprocesado_meli_vtas': df_preprocesado_meli_vtas,
    'df_preprocesado_argenprop': df_preprocesado_argenprop,
    'df_preprocesado_zonaprop': df_preprocesado_zonaprop,
    'df_preprocesado_airbnb_temp': df_preprocesado_airbnb_temp,
    'df_publicaciones_consolidadas': df_publicaciones_consolidadas,
}
for name, df in outputs.items():
    path = PROCESSED_DIR / f'{name}_v1.csv'
    df.to_csv(path, index=False)
    print('Guardado:', path.relative_to(REPO_ROOT), df.shape)

## Diccionario de datos procesado

In [ ]:
diccionario_procesado = []
for col in df_publicaciones_consolidadas.columns:
    diccionario_procesado.append({
        'columna': col,
        'tipo_dato': str(df_publicaciones_consolidadas[col].dtype),
        'significado': 'Completar definicion final.',
        'unidad': 'Completar si corresponde.',
        'fuente': 'Base consolidada desde portales inmobiliarios/Airbnb.',
        'transformacion_aplicada': 'Ver notebook 02_normalizacion_y_limpieza.',
        'ejemplo': str(df_publicaciones_consolidadas[col].dropna().iloc[0])[:120] if df_publicaciones_consolidadas[col].notna().any() else np.nan,
    })
diccionario_procesado = pd.DataFrame(diccionario_procesado)
display(diccionario_procesado)
diccionario_procesado.to_csv(PROCESSED_DIR / 'diccionario_datos_procesado_v1.csv', index=False)

## Decisiones de limpieza documentadas

In [ ]:
decisiones_limpieza = pd.DataFrame([
    {'tema': 'Faltantes estructurales', 'decision': 'Analizar por dataset antes de consolidar', 'justificacion': 'Cada fuente trae columnas distintas; unir todo desde el inicio genera nulos estructurales.'},
    {'tema': 'Airbnb', 'decision': 'Usar archivo normalizado mensual como base prioritaria', 'justificacion': 'Ya contiene barrio, comuna, amenities y variables comparables.'},
    {'tema': 'Barrios', 'decision': 'Crear barrio_norm con limpieza de tildes, aliases y subzonas', 'justificacion': 'Permite cruces territoriales sin perder el valor original.'},
    {'tema': 'Moneda', 'decision': 'Crear moneda_norm', 'justificacion': 'Evita mezclar USD, ARS y representaciones textuales.'},
    {'tema': 'Outliers', 'decision': 'No eliminar en esta etapa; se flaggean en notebook 04', 'justificacion': 'En real estate pueden existir extremos validos.'},
])
display(decisiones_limpieza)
decisiones_limpieza.to_csv(REPORTS_DIR / 'decisiones_limpieza_v1.csv', index=False)

## Cierre de limpieza inicial

La salida de este notebook es una familia de datasets preprocesados por fuente/operación y una base consolidada de columnas comunes. El análisis específico de mecanismos de ausencia se separa en el notebook 03; los valores atípicos se tratan en el notebook 04.